# 08 — Real-World Logistics & Transport Integration
Task 9: turn Task 8's assumed distance + assumed transport cost into a location-aware, vehicle-capacity-aware transport calculation, feeding back into a realistic net realization.

## Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

## Cell 2 — Load Task 8 output

In [ ]:
recommendations = pd.read_csv(
    "../data/processed/final_market_recommendations.csv"
)

recommendations.head()

In [ ]:
print(recommendations.shape)
print(recommendations.columns.tolist())

## Cell 3 — Farmer input
⚠️ These coordinates are a prototype stand-in and should eventually come from the application's location input.

In [ ]:
farmer = {
    "latitude": 20.2961,
    "longitude": 85.8245,
    "quantity_kg": 1000,
    "vehicle_capacity_kg": 1000
}

farmer

**Validate farmer input** — fail fast on nonsensical quantities before any downstream calculation runs on them.

In [ ]:
quantity_kg = farmer["quantity_kg"]
vehicle_capacity_kg = farmer["vehicle_capacity_kg"]

if quantity_kg <= 0:
    raise ValueError(
        "Quantity must be greater than zero."
    )

if vehicle_capacity_kg <= 0:
    raise ValueError(
        "Vehicle capacity must be greater than zero."
    )

## Cell 4 — Market coordinates
⚠️ These are placeholders unless you have verified coordinates — do not present invented coordinates as real market locations.

In [ ]:
market_locations = pd.DataFrame({
    "market": [
        "Market_A",
        "Market_B",
        "Market_C"
    ],
    "latitude": [
        20.30,
        20.35,
        20.40
    ],
    "longitude": [
        85.82,
        85.90,
        85.75
    ]
})

market_locations

## Cell 5 — Validate market coverage
A market recommendation should not be generated if its location is unknown.

In [ ]:
available_markets = set(
    recommendations["market"]
)

coordinate_markets = set(
    market_locations["market"]
)

missing_coordinates = (
    available_markets
    - coordinate_markets
)

print(
    "Markets without coordinates:",
    missing_coordinates
)

## Cell 6 — Merge market information

In [ ]:
df = recommendations.merge(
    market_locations,
    on="market",
    how="left"
)

df.head()

**Validate required columns** — don't let a silently-missing column produce a wrong recommendation downstream.

In [ ]:
required_columns = [
    "market",
    "latitude",
    "longitude",
    "predicted_avg"
]

missing = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

## Cell 7 — Distance calculation
Haversine formula for the initial implementation.

In [ ]:
def haversine_distance(
    lat1,
    lon1,
    lat2,
    lon2
):
    R = 6371.0

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

    return R * c

## Cell 8 — Calculate distance
Haversine gives straight-line/geodesic distance, not actual road distance. Acceptable as a transparent fallback for this prototype; an actual routing API can replace it later.

In [ ]:
df["distance_km"] = haversine_distance(
    farmer["latitude"],
    farmer["longitude"],
    df["latitude"],
    df["longitude"]
)

df[
    ["market", "distance_km"]
]

## Cell 9 — Estimate travel time

In [ ]:
average_speed_kmph = 40

df["travel_time_hours"] = (
    df["distance_km"]
    / average_speed_kmph
)

## Cell 10 — Vehicle capacity
Calculate required trips.

In [ ]:
df["number_of_trips"] = math.ceil(
    quantity_kg / vehicle_capacity_kg
)

## Cell 11 — Transport cost
Distance × cost/km × number of trips.

In [ ]:
cost_per_km = 12

df["transport_cost"] = (
    df["distance_km"]
    * cost_per_km
    * df["number_of_trips"]
)

## Cell 12 — Transport cost per kg
Useful for comparing markets on a common basis.

In [ ]:
df["transport_cost_per_kg"] = (
    df["transport_cost"]
    / quantity_kg
)

## Cell 13 — Travel-time risk
Because tomatoes are perishable, longer travel can reduce expected realization.

In [ ]:
perishability_rate_per_day = 0.03

df["travel_time_days"] = (
    df["travel_time_hours"]
    / 24
)

## Cell 14 — Perishability penalty
The 3% value is a prototype assumption, not scientifically validated spoilage data — replace with crop-specific empirical data later.

In [ ]:
df["perishability_penalty"] = (
    df["predicted_revenue"]
    * perishability_rate_per_day
    * df["travel_time_days"]
)

## Cell 15 — Recalculate net realization
Now driven by the real distance/transport values computed above, reusing `predicted_revenue` and `reliability_penalty` already carried over from Task 8's output.

In [ ]:
df["net_realization"] = (
    df["predicted_revenue"]
    - df["transport_cost"]
    - df["reliability_penalty"]
    - df["perishability_penalty"]
)

df["net_per_quintal"] = (
    df["net_realization"]
    / (quantity_kg / 100)
)

## Cell 16 — Rank markets

In [ ]:
df = df.sort_values(
    "net_realization",
    ascending=False
).reset_index(drop=True)

df["rank"] = (
    df.index + 1
)

## Cell 17 — Final recommendation

In [ ]:
best_market = df.iloc[0]

print(
    "Recommended Market:",
    best_market["market"]
)

print(
    "Distance:",
    round(best_market["distance_km"], 2),
    "km"
)

print(
    "Transport Cost:",
    round(best_market["transport_cost"], 2)
)

print(
    "Expected Net Realization:",
    round(
        best_market["net_realization"],
        2
    )
)

print(
    "Net / Quintal:",
    round(
        best_market["net_per_quintal"],
        2
    )
)

## Cell 18 — Market comparison graph

In [ ]:
top_markets = df.head(10)

plt.figure(figsize=(10, 5))

plt.bar(
    top_markets["market"],
    top_markets["net_per_quintal"]
)

plt.xlabel("Market")
plt.ylabel(
    "Expected Net Realization / Quintal"
)

plt.title(
    "Market Ranking After Logistics Cost"
)

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Cell 19 — Distance vs transport cost

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["distance_km"],
    df["transport_cost"]
)

plt.xlabel("Distance (km)")
plt.ylabel("Transport Cost")

plt.title(
    "Distance vs Transport Cost"
)

plt.tight_layout()
plt.show()

## Cell 20 — Price vs net realization
Demonstrates the core Kadak Bhav concept: highest price does not necessarily mean highest profit.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["predicted_avg"],
    df["net_per_quintal"]
)

plt.xlabel(
    "Predicted Market Price"
)

plt.ylabel(
    "Expected Net Realization / Quintal"
)

plt.title(
    "Predicted Price vs Net Realization"
)

plt.tight_layout()
plt.show()

## Cell 21 — Final recommendation table

In [ ]:
final_table = df[
    [
        "rank",
        "market",
        "predicted_avg",
        "distance_km",
        "travel_time_hours",
        "number_of_trips",
        "transport_cost",
        "reliability_penalty",
        "perishability_penalty",
        "net_realization",
        "net_per_quintal"
    ]
]

final_table.head(10)

## Cell 22 — Save results

In [ ]:
output_path = (
    "../data/processed/"
    "real_world_market_recommendations.csv"
)

df.to_csv(
    output_path,
    index=False
)

print(
    f"Saved to: {output_path}"
)

## Cell 23 — Save logistics data

In [ ]:
market_locations.to_csv(
    "../data/processed/market_locations.csv",
    index=False
)

## What's still a prototype assumption

- `market_locations` coordinates are placeholders — replace with verified real market coordinates before this feeds any real recommendation.
- Haversine distance is straight-line, not actual road distance — a routing API is the natural upgrade.
- `perishability_rate_per_day = 0.03` is still an assumption, not empirical spoilage data.
- `reliability_penalty` here is carried over as-is from Task 8's output — it still reflects whatever reliability inputs that notebook used, so its own data quality caveats still apply.